# Tutorial 2: NumPy, broadcasting and the coding framework

Copy this notebook into the `code` folder of exercise sheet 1 (next to `edf.py`), then run
`conda activate deep_learning_ex_1` and `jupyter lab` in that folder.

Each part of the notebook belongs to a part of the slides.

- **Fill in the shapes** in the comments `# shape: (fill in)`.
- Write the code. You can compare with `print(X.shape)` afterwards.
- Every task ends with a check that prints `correct` when your answer is right. Otherwise it
  prints what is still wrong, for example a wrong shape.
- In the **predict** tasks you replace each `...` with the shape you expect, or `"error"`.
- Tasks marked **now** are done in the tutorial. The tasks marked **more practice** are for later.

Run the next cell first. It imports NumPy and defines the checks for the tasks.

In [2]:
# Setup: NumPy and the checks for the tasks (no need to read this cell, just run it)
import numpy as np
np.random.seed(0)
np.set_printoptions(precision=3, suppress=True)

def compare(predictions, env):
    # predictions: {expression: your predicted shape or "error"}
    for expr, guess in predictions.items():
        try:
            real = np.shape(eval(expr, env))
        except Exception:
            real = "error"
        mark = "correct" if guess == real else "not correct yet"
        print(f"{expr:36s} you: {str(guess):14s} NumPy: {str(real):14s} {mark}")

def _problem(name, got, want):
    # returns None if got equals want, otherwise a sentence that says what is wrong
    want = np.asarray(want)
    if got is ...:
        return f"{name} is not filled in yet"
    if got is None:
        return f"{name} is None"
    try:
        shape = np.shape(got)
    except Exception:
        return f"{name} is not an array"
    if shape != want.shape:
        return f"{name} has shape {shape}, which is not the right shape"
    try:
        ok = np.allclose(got, want)
    except Exception:
        ok = False
    if not ok:
        return f"{name} has the right shape, but not the right values"
    return None

def _cell():
    # the source of the cell that is running now (None outside Jupyter)
    try:
        return get_ipython().user_ns["_ih"][-1]
    except Exception:
        return None

def _line(name):
    # the code (without comment) of the line that assigns name in the running cell
    cell = _cell()
    if cell is None:
        return None
    for line in cell.splitlines():
        code = line.split("#")[0]
        if code.replace(" ", "").startswith(name + "="):
            return code.split("=", 1)[1]
    return None

def _no_loops():
    cell = _cell()
    if cell is not None and any(l.split("#")[0].lstrip().startswith(("for ", "while ")) for l in cell.splitlines()):
        return "the task asks for no loops"
    return None

def _method(name, ok, text):
    # ok(code) says if the line that computes name does what the task asks
    code = _line(name)
    if code is None or ok(code):
        return None
    return f"{name} {text}"

def check_all(*items, methods=()):
    # items: (name, your array, the right array), methods: sentences from _method and _no_loops
    problems = [p for p in (_problem(*item) for item in items) if p]
    problems += [m for m in methods if m]
    if not problems:
        print("correct")
        return
    print("not correct yet")
    for p in problems:
        print("   ", p)

def check_1_1(X):
    check_all(("X", X, imgs.reshape(64, -1)))

def check_1_3(corners, corners_flat):
    check_all(("corners", corners, imgs[:, :14, :14]),
              ("corners_flat", corners_flat, imgs[:, :14, :14].reshape(64, -1)))

def check_1_4(mean_pixel):
    check_all(("mean_pixel", mean_pixel, imgs.mean(axis=(1, 2))[:, None]))

def check_1_5(Y, y_again):
    check_all(("Y", Y, y[:, None]), ("y_again", y_again, y))

def check_2_2(centered):
    check_all(("centered", centered, imgs - imgs.mean(axis=0)))

def check_2_3(scaled):
    check_all(("scaled", scaled, imgs / imgs.max(axis=(1, 2), keepdims=True)))

def check_2_4(squared_error):
    y_hat, t = np.random.randn(5, 1), np.random.randn(5)
    try:
        out = squared_error(y_hat, t)
    except Exception as e:
        print("not correct yet")
        print("    the check calls squared_error(y_hat, y) with y_hat (5, 1) and y (5,)")
        print("    squared_error(y_hat, y) raised", type(e).__name__, e)
        return
    problem = _problem("squared_error(y_hat, y)", out, (y_hat - t[:, None]) ** 2)
    if problem is None:
        print("correct")
        return
    print("not correct yet")
    print("    the check calls squared_error(y_hat, y) with y_hat (5, 1) and y (5,)")
    print("   ", problem)

def check_2_5(table, a, c):
    check_all(("table", table, np.outer(a, c)))

def check_3_2(H):
    Z = X @ A + b
    check_all(("H", H, 1 / (1 + np.exp(-Z))))

def check_4_2(grad_A_single, x, g):
    check_all(("grad_A_single", grad_A_single, np.outer(x, g)), methods=[_no_loops()])

def check_4_3(stack, grad_A_sum, grad_A, G):
    right = np.einsum('bd,bk->dk', X, G)
    check_all(("stack", stack, np.einsum('bd,bk->bdk', X, G)),
              ("grad_A_sum", grad_A_sum, right),
              ("grad_A", grad_A, right),
              methods=[_method("grad_A_sum", lambda c: "stack" in c, "should be the sum of stack"),
                       _method("grad_A", lambda c: ("@" in c or "matmul" in c) and "stack" not in c
                               and "grad_A_sum" not in c, "should be one matrix product of X and G with @"),
                       _no_loops()])

def check_5_1(Square):
    # runs a small network with your Square node and compares the value and the gradient with x ** 2 and 2 x
    import edf
    rng = np.random.RandomState(0)
    Xs, ys = 0.5 * rng.randn(6, 4), rng.randint(0, 2, size=6)
    try:
        edf.clear_compgraph()
        x = edf.Input()
        x.value = Xs
        y = edf.Input()
        y.value = ys
        p1, p2 = edf.AffineParams(4, 3), edf.AffineParams(3, 2)
        a1 = edf.Affine(p1, x)
        sq = Square(a1)
        L = edf.CrossEntropyLoss(edf.Softmax(edf.Affine(p2, sq)), y)
        edf.Forward()
        edf.Backward(L)
    except Exception as e:
        print("not correct yet")
        if "ellipsis" in str(e) or any(getattr(c, "value", None) is ... for c in edf.CompNodes):
            print("    Square is not filled in yet")
        else:
            print("    Square raised", type(e).__name__, e)
        return
    check_all(("self.value of forward", sq.value, a1.value ** 2),
              ("the gradient for x", a1.grad, 2 * a1.value * sq.grad))

## Part 1. Arrays for a batch of data

`imgs` is a batch of 64 grayscale images of 28x28 pixels, `y` holds one label per image.

In [ ]:
imgs = np.random.rand(64, 28, 28)
y = np.random.randint(0, 10, size=64)

### Task 1.1 (more practice) *(slide 3)*

Flatten the images into a matrix `X` with one row per image, so every image becomes
a row of 28 x 28 = 784 pixels.

In [ ]:
# imgs is (64, 28, 28)
X = ...   # shape: (fill in)
check_1_1(X)

### Task 1.2 (predict, now) *(slide 4)*

For each expression, replace `...` with the shape you expect, written as a tuple, for example
`(64, 784)`. Run the cell to compare with NumPy. The first line is an example.

In [ ]:
X = imgs.reshape(64, -1)    # as in task 1.1
# X is (64, 784), imgs is (64, 28, 28)
compare({
    "X[0]":              (784,),       # example
    "X[16:32]":          ...,          # your answers from here on
    "X[:, 100]":         ...,
    "X[5:6]":            ...,
    "X[::4]":            ...,
    "imgs[:, ::2, :]":   ...,
    "imgs[0, 14]":       ...,
    "imgs[:, 14, :]":    ...,
    "X.sum(axis=0)":     ...,
}, globals())

### Task 1.3 (more practice) *(slide 4)*

Cut the top left corner of 14x14 pixels out of every image and store it in `corners`.
Flatten every corner into one row and store the result in `corners_flat`.

In [ ]:
# imgs is (64, 28, 28)
corners = ...        # shape: (fill in)
corners_flat = ...   # shape: (fill in)
check_1_3(corners, corners_flat)

### Task 1.4 (more practice) *(slide 6)*

Compute the mean pixel value of every image and store it in `mean_pixel`, as a column with
one row per image. Sheet 1 does the same for MNIST in `x_train_mean`.

In [ ]:
# imgs is (64, 28, 28), X is (64, 784)
mean_pixel = ...   # shape: (fill in)
check_1_4(mean_pixel)

### Task 1.5 (more practice) *(slide 7)*

Turn the labels `y` into a column `Y`. Turn `Y` back into a vector `y_again` with one axis.

In [ ]:
# y is (64,)
Y = ...         # shape: (fill in)
y_again = ...   # shape: (fill in)
check_1_5(Y, y_again)

## Part 2. Broadcasting

In [ ]:
X = imgs.reshape(64, -1)   # as in task 1.1, so that the next parts also work without it

### Task 2.1 (predict, now) *(slide 9)*

For each expression, replace `...` with the shape of the result, or with `"error"` if the
shapes do not fit. Run the cell to compare with NumPy. The first line is an example.

In [ ]:
# X is (64, 784), imgs is (64, 28, 28), y is (64,)
compare({
    "X * 2":                         (64, 784),    # example
    "X + np.ones(784)":              ...,          # your answers from here on
    "X + np.ones(64)":               ...,
    "X + np.ones((64, 1))":          ...,
    "X - X.mean(axis=0)":            ...,
    "imgs * np.ones((28, 1))":       ...,
    "y[:, None] - y[None, :]":       ...,
}, globals())

### Task 2.2 (more practice) *(slide 10)*

Center the images, a common preprocessing step.

1. Compute `mean_image`, the average of each pixel across all 64 images. It is one image of 28x28.
2. Compute `centered` by subtracting `mean_image` from every image. Afterwards every pixel has
   mean 0 across the 64 images.

In [ ]:
# imgs is (64, 28, 28)
mean_image = ...   # shape: (fill in)
centered = ...     # shape: (fill in)
check_2_2(centered)

### Task 2.3 (more practice) *(slide 6)*

Scale every image so that its largest pixel is 1.

1. Compute `max_pixel`, the largest pixel of every image, with a shape that broadcasts against `imgs`.
2. Compute `scaled` by dividing every image by its own `max_pixel`.

In [ ]:
# imgs is (64, 28, 28)
max_pixel = ...   # shape: (fill in)
scaled = ...      # shape: (fill in)
check_2_3(scaled)

### Task 2.4 (now) *(slide 11)*

`squared_error(y_hat, y)` should return the squared error `(y_hat - y) ** 2` of every sample, with
the shape of `y_hat`, `(B, 1)`. The function runs without an error message, but the result is
wrong. Fill in the shape of `d`. Find and fix the bug.

In [ ]:
def squared_error(y_hat, y):
    # y_hat shape: (B, 1), y shape: (B,)
    d = y_hat - y      # shape: (fill in)
    return d ** 2

check_2_4(squared_error)

### Task 2.5 (more practice) *(slide 12)*

`a` has 3 numbers and `c` has 2 numbers. Compute `table`, the matrix with
`table[i, j] = a[i] * c[j]` (3 rows, 2 columns), without loops. Use `None` to add the axes you need.

In [ ]:
a = np.array([1., 2., 3.])     # (3,)
c = np.array([10., 20.])       # (2,)
table = ...   # shape: (fill in)
check_2_5(table, a, c)

## Part 3. Multiplying multi-dimensional arrays

`A` and `b` are the parameters of a layer with 784 inputs and 10 outputs.
`S` and `T` are stacks of 8 matrices.

In [ ]:
A = np.random.randn(784, 10) * 0.05
b = np.random.randn(10)
S = np.random.randn(8, 3, 4)
T = np.random.randn(8, 4, 5)

### Task 3.1 (predict, now) *(slide 14)*

For each expression, replace `...` with the shape of the result, or with `"error"` if the
shapes do not fit. Run the cell to compare with NumPy. The first line is an example.

In [ ]:
# X is (64, 784), A is (784, 10), S is (8, 3, 4), T is (8, 4, 5)
compare({
    "np.ones((5, 3)) @ np.ones((3, 2))": (5, 2),  # example
    "X @ A":                         ...,          # your answers from here on
    "A.T @ X.T":                     ...,
    "S @ T":                         ...,
    "S @ np.ones((4, 5))":           ...,
    "S @ np.ones((2, 4, 5))":        ...,
    "S[:, :1, :] @ T[:, :, :1]":     ...,
}, globals())

### Task 3.2 (more practice) *(slide 5)*

Compute the forward pass of the layer for the whole batch, `Z = X A + b` and
`H = sigmoid(Z) = 1 / (1 + exp(-Z))`.

In [ ]:
# X is (64, 784), A is (784, 10), b is (10,)
Z = ...   # X A + b, shape: (fill in)
H = ...   # sigmoid, shape: (fill in)
check_3_2(H)

## Part 4. The weight gradient of a layer

The layer is `Y = X @ A + b`. `G` is the gradient of the loss with respect to `Y`, from the layer above.
It has the shape of `Y`, one row `g` per sample.

A name `grad_...` is the gradient of the loss with respect to an array, and it has the shape of that
array. For example, `grad_A` is the gradient with respect to `A`, with the shape of `A`.

In [ ]:
G = np.random.randn(64, 10)    # (64, 10)
i = 0                          # one sample, any index from 0 to 63 works
x = X[i]                       # sample i, (784,)
g = G[i]                       # the gradient for sample i, (10,)

### Task 4.1 (more practice) *(slide 16)*

`grad_A_single` is the gradient of the loss
with respect to the whole matrix `A`, for the single sample `x`. The entry `A[d, k]` connects input `d`
to output `k`, so its gradient is `x[d] * g[k]`. Compute it for all entries at once, as the outer
product of `x` and `g`.

In [ ]:
# x is (784,), g is (10,)
grad_A_single = ...   # gradient for A, one sample, shape: (fill in)
check_4_2(grad_A_single, x, g)

### Task 4.2 (now) *(slide 17)*

Every sample `i` gives an outer product of `X[i]` and `G[i]`, as in task 4.1. We stack these 64
matrices along the batch axis into one array `stack`, so that `stack[i]` is the outer product of
sample `i`. The gradient for `A` for the whole batch is the sum of the stack over the samples.
Compute it without loops, in three steps.

1. Compute `stack` with broadcasting.
2. Sum `stack` over the samples into `grad_A_sum`.
3. Compute the same gradient `grad_A` with one matrix product.

In [ ]:
# X is (64, 784), G is (64, 10)
stack = ...        # shape: (fill in)
grad_A_sum = ...   # shape: (fill in)
grad_A = ...       # gradient for A, batch, with @, shape: (fill in)
check_4_3(stack, grad_A_sum, grad_A, G)

## Part 5. The Educational Framework (EDF)

Slide 24.

In [ ]:
import edf

### Task 5.1 (now) *(slide 24)*

Write the node `Square`, `y[b, k] = x[b, k] ** 2` for every entry. `forward` computes `self.value`.
`backward` calls `addgrad` once for `self.x`. The check runs a small network with your node.

In [ ]:
class Square(edf.CompNode):
    def __init__(self, x):
        edf.CompNodes.append(self)
        self.x = x      # x.value is (B, K)

    def forward(self):
        self.value = ...   # shape: (fill in)

    def backward(self):
        # your code: one addgrad call for self.x, with the shape of x.value
        pass

check_5_1(Square)